# Netflix Content Analysis

Exploratory analysis of the Netflix catalog: content mix, top producing countries, release trends, genres and audience ratings.

**Dataset:** `data/Netflix.csv` (8,790 titles, 10 columns)

## 1. Load the data

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs('images', exist_ok=True)

df = pd.read_csv('data/Netflix.csv')
df.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


## 2. Data cleaning

In [2]:
# Identifying missing values.
df.isnull().sum()

show_id         0
type            0
title           0
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64

In [3]:
# Removing duplicate records
print('Duplicate rows:', df.duplicated().sum())
df = df.drop_duplicates()
print('Rows after de-duplication:', len(df))

Duplicate rows: 0
Rows after de-duplication: 8790


In [4]:
# The dataset uses the placeholder 'Not Given' instead of nulls.
for col in ['director', 'country']:
    print(col, '->', (df[col] == 'Not Given').sum(), "rows marked 'Not Given'")

director -> 2588 rows marked 'Not Given'
country -> 287 rows marked 'Not Given'


In [5]:
# Standardizing columns Country, Rating, and Type.
# 'Not Given' placeholders are converted to 'Unknown' so they are not counted as a real country.
df['country'] = df['country'].fillna('Unknown').replace('Not Given', 'Unknown')
df['primary country'] = df['country'].apply(lambda x: x.split(',')[0].strip())

df['rating'] = df['rating'].fillna('Unknown')
df['primary rating'] = df['rating'].apply(lambda x: x.split(',')[0].strip())

df['type'] = df['type'].fillna('Unknown')
df['primary type'] = df['type'].apply(lambda x: x.split(',')[0].strip())
df.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in,primary country,primary rating,primary type
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries,United States,PG-13,Movie
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",France,TV-MA,TV Show
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries",United States,TV-MA,TV Show
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies",Brazil,TV-PG,Movie
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies",United States,TV-MA,Movie


In [6]:
df = df.drop(columns=['country', 'rating', 'type'])
df.head()

,show_id,title,director,date_added,release_year,duration,listed_in,primary country,primary rating,primary type
0,s1,Dick Johnson Is Dead,Kirsten Johnson,9/25/2021,2020,90 min,Documentaries,United States,PG-13,Movie
1,s3,Ganglands,Julien Leclercq,9/24/2021,2021,1 Season,"Crime TV Shows, International TV Shows, TV Act...",France,TV-MA,TV Show
2,s6,Midnight Mass,Mike Flanagan,9/24/2021,2021,1 Season,"TV Dramas, TV Horror, TV Mysteries",United States,TV-MA,TV Show
3,s14,Confessions of an Invisible Girl,Bruno Garotti,9/22/2021,2021,91 min,"Children & Family Movies, Comedies",Brazil,TV-PG,Movie
4,s8,Sankofa,Haile Gerima,9/24/2021,1993,125 min,"Dramas, Independent Movies, International Movies",United States,TV-MA,Movie


In [7]:
# Exporting the cleaned dataset for analysis.
df.to_csv('data/Netflix_Cleaned.csv', index=False)

In [8]:
# Loading the cleaned dataset
df2 = pd.read_csv('data/Netflix_Cleaned.csv')
df2.head()

,show_id,title,director,date_added,release_year,duration,listed_in,primary country,primary rating,primary type
0,s1,Dick Johnson Is Dead,Kirsten Johnson,9/25/2021,2020,90 min,Documentaries,United States,PG-13,Movie
1,s3,Ganglands,Julien Leclercq,9/24/2021,2021,1 Season,"Crime TV Shows, International TV Shows, TV Act...",France,TV-MA,TV Show
2,s6,Midnight Mass,Mike Flanagan,9/24/2021,2021,1 Season,"TV Dramas, TV Horror, TV Mysteries",United States,TV-MA,TV Show
3,s14,Confessions of an Invisible Girl,Bruno Garotti,9/22/2021,2021,91 min,"Children & Family Movies, Comedies",Brazil,TV-PG,Movie
4,s8,Sankofa,Haile Gerima,9/24/2021,1993,125 min,"Dramas, Independent Movies, International Movies",United States,TV-MA,Movie


## 3. Content distribution: Movies vs TV Shows

In [9]:
# Calculating the total number of Movies and TV Shows.
type_counts = df2['primary type'].value_counts()
print(type_counts)

primary type
Movie      6126
TV Show    2664
Name: count, dtype: int64


In [10]:
# Creating visualizations for content distribution.
ax = type_counts.plot(kind='bar', color=['#E50914', 'black'])
plt.title('Content Distribution: Movies vs TV Shows')
plt.xlabel('Type')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('images/content_distribution.png', dpi=150)
plt.show()

In [11]:
# Comparing content proportions.
type_percentages = df2['primary type'].value_counts(normalize=True) * 100
print(type_percentages.round(1))

primary type
Movie      69.7
TV Show    30.3
Name: proportion, dtype: float64


**Key finding:** Movies account for about 70% of the catalog, while TV Shows make up about 30%.

## 4. Top content-producing countries

In [12]:
# Extracting and cleaning country information (primary production country per title).
df2_country = df2[df2['primary country'] != 'Unknown'].copy()
print('Titles with a known country:', len(df2_country), 'of', len(df2))

Titles with a known country: 8503 of 8790


In [13]:
# Calculating content count by country
country_counts = df2_country['primary country'].value_counts()
country_counts.head(15)

primary country
United States     3240
India             1057
United Kingdom     638
Pakistan           421
Canada             271
Japan              259
South Korea        214
France             213
Spain              182
Mexico             138
Egypt              123
Australia          114
Turkey             112
Nigeria            105
Germany            104
Name: count, dtype: int64

In [14]:
# Identifying top content-producing countries
top_10_countries = country_counts.head(10)
print(top_10_countries)

primary country
United States     3240
India             1057
United Kingdom     638
Pakistan           421
Canada             271
Japan              259
South Korea        214
France             213
Spain              182
Mexico             138
Name: count, dtype: int64


In [15]:
# Creating charts and rankings
top_10_countries.sort_values().plot(kind='barh', color='#E50914')
plt.title('Top 10 Content-Producing Countries')
plt.xlabel('Number of Titles')
plt.tight_layout()
plt.savefig('images/top_countries.png', dpi=150)
plt.show()

In [16]:
# Share of titles from the top two countries
top2_share = country_counts.head(2).sum() / len(df2_country) * 100
print(f"{country_counts.index[0]} + {country_counts.index[1]}: {top2_share:.1f}% of titles with a known country")

United States + India: 50.5% of titles with a known country


**Key finding:** The United States and India together account for about 50% of titles with a known primary country (3,240 and 1,057 titles), making them the biggest production hubs in the catalog. The United Kingdom (638) and Pakistan (421) follow.

## 5. Content releases by year

In [17]:
# Organizing content by release year
yearly = df2.groupby('release_year').size()
yearly.tail(15)

release_year
2007      88
2008     135
2009     152
2010     192
2011     185
2012     236
2013     286
2014     352
2015     555
2016     901
2017    1030
2018    1146
2019    1030
2020     953
2021     592
dtype: int64

In [18]:
# Identifying growth and decline trends
yearly_sorted = yearly.sort_index()
peak_year = yearly_sorted.idxmax()
print(f"Peak release year: {peak_year} with {yearly_sorted.max()} titles")

Peak release year: 2018 with 1146 titles


In [19]:
# Creating trend visualization (1990 onward for readability)
yearly_sorted[yearly_sorted.index >= 1990].plot(kind='line', marker='o', color='#E50914')
plt.title('Content Releases by Year')
plt.xlabel('Release Year')
plt.ylabel('Number of Titles')
plt.grid(True)
plt.tight_layout()
plt.savefig('images/releases_by_year.png', dpi=150)
plt.show()

**Key finding:** Releases grew steadily from 2015 and peaked in 2018 with 1,146 titles, then declined. The drop after 2018 may reflect slower content acquisition, and 2021 is likely incomplete because the dataset stops in 2021.

## 6. Ratings and genres

In [20]:
# Analyzing rating categories
rating_counts = df2['primary rating'].value_counts()
print(rating_counts)

primary rating
TV-MA       3205
TV-14       2157
TV-PG        861
R            799
PG-13        490
TV-Y7        333
TV-Y         306
PG           287
TV-G         220
NR            79
G             41
TV-Y7-FV       6
NC-17          3
UR             3
Name: count, dtype: int64


In [21]:
# Identifying most common genres
df_genres = df2.assign(listed_in=df2['listed_in'].str.split(',')).explode('listed_in')
df_genres['listed_in'] = df_genres['listed_in'].str.strip()
genre_counts = df_genres['listed_in'].value_counts()
print(genre_counts.head(10))

listed_in
International Movies        2752
Dramas                      2426
Comedies                    1674
International TV Shows      1349
Documentaries                869
Action & Adventure           859
TV Dramas                    762
Independent Movies           756
Children & Family Movies     641
Romantic Movies              616
Name: count, dtype: int64


In [22]:
# Top genres chart
genre_counts.head(10).sort_values().plot(kind='barh', color='#E50914')
plt.title('Top 10 Genres')
plt.xlabel('Number of Titles')
plt.tight_layout()
plt.savefig('images/top_genres.png', dpi=150)
plt.show()

In [23]:
# Comparing ratings across content types
rating_by_type = pd.crosstab(df2['primary type'], df2['primary rating'])
rating_by_type

primary rating,G,NC-17,NR,PG,PG-13,R,TV-14,TV-G,TV-MA,TV-PG,TV-Y,TV-Y7,TV-Y7-FV,UR
primary type,,,,,,,,,,,,,,
Movie,41,3,75,287,490,797,1427,126,2062,540,131,139,5,3
TV Show,0,0,4,0,0,2,730,94,1143,321,175,194,1,0


In [24]:
# Stacked bar chart of ratings by content type
rating_by_type.T.plot(kind='bar', stacked=True, figsize=(9, 5), color=['#E50914', 'black'])
plt.title('Ratings Distribution by Content Type')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.legend(title='Type')
plt.tight_layout()
plt.savefig('images/ratings_by_type.png', dpi=150)
plt.show()

**Key finding:** TV-MA is the most common rating (3,205 titles, about 36%), followed by TV-14 (2,157). The catalog skews toward mature audiences across both Movies and TV Shows.

## 7. Export analysis tables

In [25]:
df2_country.to_csv('data/netflix_by_country.csv', index=False)
df_genres.to_csv('data/netflix_by_genre.csv', index=False)